In [1]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages
from langchain.chat_models import init_chat_model
from langchain_tavily import TavilySearch
from langgraph.graph import StateGraph, START
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver

from dotenv import load_dotenv
load_dotenv()

llm = init_chat_model("openai:gpt-4.1")

class State(TypedDict):
    messages:Annotated[list, add_messages]
    name: str
    release_date: str

In [2]:
# 도구에서 상태 업데이트하기 : 파라미터

# human_assistance 도구 확장: interrupt()로 사람에게 정보 검토 요청

from langchain_core.messages import ToolMessage
from langchain_core.tools import InjectedToolCallId, tool
from langgraph.types import Command, interrupt

@tool
def human_assistance(name:str, release_date:str, tool_call_id: Annotated[str, InjectedToolCallId]):
    """사람의 확인이 필요한 정보를 검토받는 도구입니다."""
    human_response = interrupt({"question":"이 정보가 맞나요?", "name":name, "release_date":release_date})

    # 사람이 "Yes"로 응답한 경우 기존 정보를 그대로 사용
    if human_response.get("correct","").lower().startswith("y"):
        verified_name = name
        verified_date = release_date
        response = "정보가 정확하다고 확인됨."

    # 수정이 필요한 경우 사람이 제공한 값을 사용
    else:
        verified_name = human_response.get("name",name)
        verified_date = human_response.get("release_date", release_date)
        response = f"사람이 수정한 정보: {human_response}"

    # 상태에 name, release_date 저장 + 메시지도 함께 반환
    state_update = {"name":verified_name, "release_date":verified_date,
                    "messages":[ToolMessage(response, tool_call_id=tool_call_id)]}
    return Command(update=state_update) # 상태 갱신

In [3]:
# 챗봇 그래프 구성하기

search_tool = TavilySearch(max_results=2)
tools = [search_tool, human_assistance]
llm_with_tools = llm.bind_tools(tools)

def chatbot(state:State):
    message = llm_with_tools.invoke(state["messages"])
    assert(len(message.tool_calls) <= 1)
    return {"messages":[message]}

graph_builder = StateGraph(State)
graph_builder.add_node("chatbot", chatbot)
tool_node = ToolNode(tools=tools)
graph_builder.add_node("tools", tool_node)

graph_builder.add_conditional_edges("chatbot", tools_condition)
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge(START, "chatbot")

memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)

In [4]:
user_input = (
    "랭그래프가 언제 출시되었는지 찾아줄래요?"
    "결과를 찾으면 human_assistance 도구를 사용해서 사람에게 확인해줘.")
config = {"configurable": {"thread_id":"1"}}
events = graph.stream({"messages":[{"role":"user","content":user_input}]}, config, stream_mode="values")

for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

================================ Human Message =================================

랭그래프가 언제 출시되었는지 찾아줄래요?결과를 찾으면 human_assistance 도구를 사용해서 사람에게 확인해줘.
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_EfMUrt5K3LjtI4aDCCyytCvy)
 Call ID: call_EfMUrt5K3LjtI4aDCCyytCvy
  Args:
    query: 랭그래프 출시일
================================= Tool Message =================================
Name: tavily_search

{"query": "랭그래프 출시일", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://kr.linkedin.com/pulse/langgraph-v10-coming-what-means-ai-development-zoftcares-2e5ac?tl=ko", "title": "LangGraph v1.0이 출시됩니다: 이것이 AI 개발에 의미하는 바", "content": "9월 2일, LangChain은 LangGraph v1.0의 알파 버전을 공개하며 상태 기반 AI 워크플로우 오케스트레이션에 중요한 이정표를 세웠습니다.", "score": 0.7636429, "raw_content": null, "id": "5fef1f-00"}, {"url": "https://velog.io/@hsgogo/%EB%9E%AD%EA%B7%B8%EB%9E%98%ED%94%84", "title": "랭그래프", "content": "hsgogo.log\n\n# 랭그래프\

In [5]:
snapshot = graph.get_state(config)
print(snapshot.next)

('tools',)


In [6]:
from langgraph.types import Command

human_command = Command(
    resume={"name":"랭그래프", "release_date":"2024년 1월 17일"},
)
events = graph.stream(human_command, config, stream_mode="values")
for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

================================== Ai Message ==================================
Tool Calls:
  human_assistance (call_ZFC7kJMoJ6O1iAyTAtl3ZdNg)
 Call ID: call_ZFC7kJMoJ6O1iAyTAtl3ZdNg
  Args:
    name: LangGraph (랭그래프)
    release_date: 2023-09-02
================================= Tool Message =================================
Name: human_assistance

사람이 수정한 정보: {'name': '랭그래프', 'release_date': '2024년 1월 17일'}
================================== Ai Message ==================================

사람의 확인 결과, 랭그래프(LangGraph)의 공식 출시일은 2024년 1월 17일입니다.


In [7]:
# 인간의 수정이 잘 반영되었는지 확인

snapshot = graph.get_state(config)
{k: v for k, v in snapshot.values.items() if k in ("name", "release_date")}

{'name': '랭그래프', 'release_date': '2024년 1월 17일'}

In [8]:
# update_state()로 상태의 특정 키를 직접 덮어쓰기

graph.update_state(config, {"name":"LangGraph (library)"})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1d1d3c-a1a0-63f3-8006-0c025061c43e'}}

# 시간 이동 Time Travel

In [9]:
# 그래프의 상태 기록 전체를 가져옵니다(지금까지의 실행 히스토리)

history = graph.get_state_history(config)
to_replay = None
for state in history:
    # 각 상태에서 메시지 개수와 다음 노드를 출력합니다.
    print("Messages:", len(state.values["messages"]), "Next:", state.next)
    print("-" * 80)

    # 메시지가 6개인 상태를 선택합니다(임의 기준으로 선택)
    if len(state.values["messages"]) == 3:
        # 나중에 되감기(replay)할 상태로 저장합니다
        to_replay = state

Messages: 6 Next: ()
--------------------------------------------------------------------------------
Messages: 6 Next: ()
--------------------------------------------------------------------------------
Messages: 5 Next: ('chatbot',)
--------------------------------------------------------------------------------
Messages: 4 Next: ('tools',)
--------------------------------------------------------------------------------
Messages: 3 Next: ('chatbot',)
--------------------------------------------------------------------------------
Messages: 2 Next: ('tools',)
--------------------------------------------------------------------------------
Messages: 1 Next: ('chatbot',)
--------------------------------------------------------------------------------
Messages: 0 Next: ('__start__',)
--------------------------------------------------------------------------------


In [10]:
# 특정 시점에서 실행 재개하기

print(to_replay.next) # -> ('chatbot',)
print(to_replay.config) # -> {'configurable': {'thread_id':'1', 'checkpoint_id':'1f0751f2-798f-...'}}

# 저장된 시점으로 되감아 실행
for event in graph.stream(None, to_replay.config, stream_mode="values"):
    if "messages" in event:
        event["messages"][-1].pretty_print()

('chatbot',)
{'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1d1d3c-92a5-60bc-8002-6f731cc15027'}}
================================= Tool Message =================================
Name: tavily_search

{"query": "랭그래프 출시일", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://kr.linkedin.com/pulse/langgraph-v10-coming-what-means-ai-development-zoftcares-2e5ac?tl=ko", "title": "LangGraph v1.0이 출시됩니다: 이것이 AI 개발에 의미하는 바", "content": "9월 2일, LangChain은 LangGraph v1.0의 알파 버전을 공개하며 상태 기반 AI 워크플로우 오케스트레이션에 중요한 이정표를 세웠습니다.", "score": 0.7636429, "raw_content": null, "id": "5fef1f-00"}, {"url": "https://velog.io/@hsgogo/%EB%9E%AD%EA%B7%B8%EB%9E%98%ED%94%84", "title": "랭그래프", "content": "hsgogo.log\n\n# 랭그래프\n\n현서·2025년 9월 8일\n\n랭그래프\n\n## AI Agent\n\n목록 보기\n\n2/8\n\npost-thumbnail\n\n## 1. 랭그래프(LangGraph)\n\n 랭그래프는 LangChain 생태계에서 에이전트나 RAG 시스템을 단계별로 구성하고 실행할 수 있게 해주는 그래프 기반 오케스트레이션 프레임워크이다.\n 기존 RAG가 직선형 파이프라인이었다면, 랭그래프는 노드(작업 단위)와 

In [11]:
snapshot = graph.get_state(config)
print(snapshot.next)

('tools',)
